**Ex4.1: Automatic Speech Recognition (ASR) System**

In [75]:
from deepspeech import Model
import librosa as lr
import numpy as np
import string
import unicodedata
import csv
from scipy.signal import butter, lfilter
from prettytable import PrettyTable
import threading
import time
from termcolor import colored

# Step 1: Initialize DeepSpeech models for English, Italian, and Spanish
def initialize_models():
    models = {}

    # English model
    scorer_en = "Models/EN/deepspeech-0.9.3-models.scorer"
    model_path_en = "Models/EN/deepspeech-0.9.3-models.pbmm"
    model_en = Model(model_path_en)
    model_en.enableExternalScorer(scorer_en)
    models["English"] = model_en

    # Italian model
    scorer_it = "Models/IT/kenlm_it.scorer"
    model_path_it = "Models/IT/output_graph_it.pbmm"
    model_it = Model(model_path_it)
    model_it.enableExternalScorer(scorer_it)
    models["Italian"] = model_it

    # Spanish model
    scorer_es = "Models/ES/kenlm_es.scorer"
    model_path_es = "Models/ES/output_graph_es.pbmm"
    model_es = Model(model_path_es)
    model_es.enableExternalScorer(scorer_es)
    models["Spanish"] = model_es

    return models

# Step 2: Preprocess audio using ZCR to detect and reduce noise
def preprocess_audio_zcr(audio, sample_rate, threshold=0.1):
    zcr = lr.feature.zero_crossing_rate(audio, frame_length=2048, hop_length=512)
    zcr_mean = np.mean(zcr)

    if zcr_mean < threshold:
        audio = audio * 1.5  # Apply gain for low ZCR
    else:
        audio = audio * 0.8  # Reduce gain for high ZCR

    return np.clip(audio, -1.0, 1.0)  # Clip audio to valid range

# Step 3: Recognize speech from a single audio file
def recognize_audio(file_path, model, desired_sample_rate=16000):
    try:
        # Load the audio file using librosa
        audio, sample_rate = lr.load(file_path, sr=desired_sample_rate)

        # Preprocess audio using ZCR
        audio = preprocess_audio_zcr(audio, sample_rate)

        # Scale audio to 16-bit range
        audio = (audio * 32767).astype(np.int16)

        # Perform speech recognition
        hypothesis = model.stt(audio)
        return hypothesis
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return "Recognition Failed"

# Step 4: Preprocess text to handle accents, punctuation, and case
def preprocess_text(text):
    text = text.replace("-", " ").lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    return text

# Step 5: Compute Word Error Rate (WER)
def compute_wer(reference, hypothesis):
    ref_words = preprocess_text(reference).split()
    hyp_words = preprocess_text(hypothesis).split()

    dp = [[0 for _ in range(len(hyp_words) + 1)] for _ in range(len(ref_words) + 1)]

    for i in range(len(ref_words) + 1):
        for j in range(len(hyp_words) + 1):
            if i == 0:
                dp[i][j] = j
            elif j == 0:
                dp[i][j] = i
            elif ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    wer = dp[len(ref_words)][len(hyp_words)] / len(ref_words) * 100
    return wer

# Spinner animation
def spinner_animation(stop_event):
    spinner = ["   ", ".  ", ".. ", "..."]
    idx = 0
    while not stop_event.is_set():
        print(f"\rGenerating table, please wait{spinner[idx]}", end="")
        idx = (idx + 1) % len(spinner)
        time.sleep(0.5)
    print("\rTable generation complete!             ")

# Step 6: Process all files from the CSV
def process_csv_transcriptions_with_spinner(csv_file_path, models):
    table = PrettyTable()
    table.field_names = ["Language", "File", "WER (%)"]
    table.align = "l"  # Align text to the left
    table.align["WER (%)"] = "r"  # Right align the WER column

    stop_event = threading.Event()
    spinner_thread = threading.Thread(target=spinner_animation, args=(stop_event,))
    spinner_thread.start()  # Start the spinner animation

    try:
        with open(csv_file_path, mode="r", encoding="utf-8-sig") as csvfile:
            reader = csv.DictReader(csvfile)

            for row in reader:
                language = row["Language"]
                file_path = row["File"]
                reference_text = row["Transcriptions"]

                model = models.get(language)

                if model:
                    hypothesis = recognize_audio(file_path, model)
                    if hypothesis and hypothesis != "Recognition Failed":
                        wer = compute_wer(reference_text, hypothesis)
                        wer_color = colored(f"{wer:.2f}", "green" if wer == 0.0 else "red")
                        table.add_row([language, file_path, wer_color])
                    else:
                        table.add_row([language, file_path, colored("Recognition Failed", "red")])
                else:
                    table.add_row([language, file_path, colored("No Model", "red")])

    finally:
        stop_event.set()  # Signal the spinner to stop
        spinner_thread.join()  # Wait for the spinner thread to finish

    print("\nRecognition Results:")
    print(table)

# Main execution
if __name__ == "__main__":
    models = initialize_models()
    csv_file_path = "transcriptions_03.csv"
    process_csv_transcriptions_with_spinner(csv_file_path, models)

Table generation complete!             

Recognition Results:
+----------+-----------------------------+---------+
| Language | File                        | WER (%) |
+----------+-----------------------------+---------+
| English  | Audio/EN/checkin.wav        |    0.00 |
| English  | Audio/EN/parents.wav        |   20.00 |
| English  | Audio/EN/suitcase.wav       |    0.00 |
| English  | Audio/EN/what_time.wav      |   20.00 |
| English  | Audio/EN/where.wav          |    0.00 |
| English  | Audio/EN/your_sentence1.wav |    0.00 |
| English  | Audio/EN/your_sentence2.wav |    0.00 |
| Italian  | Audio/IT/checkin_it.wav     |   25.00 |
| Italian  | Audio/IT/parents_it.wav     |   20.00 |
| Italian  | Audio/IT/suitcase_it.wav    |    0.00 |
| Italian  | Audio/IT/what_time_it.wav   |  100.00 |
| Italian  | Audio/IT/where_it.wav       |   42.86 |
| Spanish  | Audio/ES/checkin_es.wav     |   25.00 |
| Spanish  | Audio/ES/parents_es.wav     |    0.00 |
| Spanish  | Audio/ES/suitcase_es.wav